# D8 flow accumulation: tracing contributing cells

This lab starts with a direction raster and propagates upstream contributions downstream. It does not derive directions from a DEM or fill depressions.

The count includes each cell itself. Invalid cells are shown as NoData.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'implementations').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'implementations/python/d8-flow-accumulation'))
from d8_flow_accumulation import d8_flow_accumulation


## 1. A small drainage network

The direction codes use the ESRI convention: E=1, SE=2, S=4, SW=8, W=16, NW=32, N=64, NE=128. Code 0 is a terminal cell and -1 is invalid.

In [ ]:
directions = np.array([
    [-1, 4, -1, -1, -1],
    [1, 1, 1, 4, -1],
    [-1, -1, 1, 1, 0],
] , dtype=int)
accumulation = d8_flow_accumulation(directions)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(np.ma.masked_where(directions == -1, directions), cmap='tab20')
axes[0].set_title('D8 direction codes'); axes[0].set_xticks(range(directions.shape[1])); axes[0].set_yticks(range(directions.shape[0]))
im = axes[1].imshow(np.ma.masked_invalid(accumulation), cmap='viridis')
axes[1].set_title('Accumulation (including self)'); axes[1].set_xticks(range(directions.shape[1])); axes[1].set_yticks(range(directions.shape[0]))
fig.colorbar(im, ax=axes[1], label='Contributing cells'); plt.tight_layout(); plt.show()
print(accumulation)


## 2. Weighted contribution

Instead of counting each cell as one, supply a non-negative contribution such as an illustrative rainfall-excess amount. The output then sums those weights, not cell counts.

In [ ]:
weights = np.where(directions >= 0, 2.0, np.nan)
weighted = d8_flow_accumulation(directions, weights=weights)
print('Weighted accumulation:')
print(weighted)


## 3. Why a cycle is invalid

A flow-direction graph should be acyclic. Try the two-cell east-west loop below; no upstream-to-downstream topological order exists, so the implementation raises an error.

In [ ]:
try:
    d8_flow_accumulation([[1, 16]])
except ValueError as error:
    print(type(error).__name__ + ':', error)


## Investigate

1. Trace the upstream contributors to the cell with the largest count.
2. Change one direction code. Which downstream cells change?
3. Add nonuniform weights. How does the interpretation of the output change?
4. What should happen when flow points beyond the raster boundary?
5. Why is D8 not the same as D-infinity or multiple-flow-direction routing?
6. Explain why this result is not automatically discharge.